## Imports

In [248]:
import numpy as np
import pandas as pd

import math
import random
import gc

import regex as re
from rapidfuzz import fuzz
from tqdm import tqdm   


from collections import defaultdict, deque
from typing import Any

## Loading
Loading the data, perhaps via chunking, but this may cause issues with duplicate detection later on

In [2]:
"""
Loads chunks of data from csv file

Returns:
    pd.DataFrame: The selected chunk of data
"""
def load_csv_chunk(
    filepath: str,              # Path to .csv file
    chunks: int = 1,            # Number of (roughly) equal parts to split the csv into
    chunk_idx: int = 0,         # Index of the chunk to load (zero-based)
    offset: int = 1,            # Number of header lines in file
    **read_csv_kwargs           # Extra arguments passed to pd.read_csv()
):

    # Count number of data rows, excluding the header
    #   Get specified encoding, otherwise default utf-8
    with open(filepath, "r", encoding=read_csv_kwargs.get("encoding", "utf-8")) as f:
        total_rows = sum(1 for line in f) - offset

    # Chunking/Slicing Information
    chunk_size = math.ceil(total_rows / chunks)           # Size of chunk
    start_row = chunk_idx * chunk_size                     # Index of start row in actual data
    end_row = min(start_row + chunk_size, total_rows)     # Index of end row in actual data
    nrows = end_row - start_row                           # Number of rows to read
    header_row = offset - 1                               # Row index for the header information (assumed as the last header row)

    # Extract the header/columns
    columns = pd.read_csv(
        filepath,
        skiprows=header_row,
        nrows=0,
        **read_csv_kwargs
    ).columns

    # Read only the selected chunk of data
    df = pd.read_csv(
        filepath,
        skiprows=offset + start_row,
        nrows=nrows,
        names=columns,
        header=None,
        **read_csv_kwargs
    )

    return df

In [3]:
raw_sample = load_csv_chunk("UK-Sanctions-List.csv", offset=2)

C:\Users\alecz\AppData\Local\Temp\ipykernel_23480\4224360936.py:36: DtypeWarning: Columns (0: IMO number, 1: Current owner/operator (s), 2: Previous owner/operator (s), 3: Current believed flag of ship, 4: Previous flags, 5: Type of ship) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


## Exploration
Explore the fields (columns), look for data we're interested in (Name, DOB, Countries, other...)

By the end of this step, we should have an understanding of which fields are useful and which can be excluded.

In [4]:
"""
Gives useful information and examples of data in a specific column
Helps decide if it is useful for our purposes
"""
def profile_column(
        df: pd.DataFrame,                # Dataframe to analyse
        col_name: str,                   # Column identifier
        top_n: int = 5,                  # Show top_n most common values
        sampl_n: int = 10,               # Show sampl_n number of random examples
        key: list[str] | None = None     # Print Key fields to the examples
) -> None:
    
    if key is None:
        key = []
    
    col = df[col_name]
    rows = len(col)
    missg = col.isna().sum()
    non_missg = col.notna().sum()
    missg_pct = round((missg / rows) * 100, 2)
    unique = col.nunique(dropna=True)

    # Basic Info
    print("#" * 40)
    print(f"Column: {col_name}")
    print("#" * 40)
    print(f"Total rows:          {rows}")
    print(f"Data type:           {col.dtype}")
    print(f"Non-missing values:  {non_missg}")
    print(f"Missing values:      {missg}")
    print(f"Missing %:           {missg_pct}%")
    print(f"Unique values:       {unique}")
    if non_missg > 0:
        print(f"Most common value:   {col.value_counts(dropna=True).index[0]}")
        print(f"Most common count:   {col.value_counts(dropna=True).iloc[0]}")

    # Print top occurences
    print("\nTop value counts:")
    print("-" * 20)
    print(col.value_counts(dropna=False).head(top_n))


    # Build Example Data
    example_cols = key + [col_name]
    examples = (
        df[df[col_name].notna()]
        [example_cols]
        .drop_duplicates(subset=[col_name])   # drops duplicates across col_name
    )


    # Print some random examples
    print("\nRandom non-missing examples:")
    print("-" * 20)
    if len(examples) == 0:
        print("No non-missing examples available.")
    else:
        print(
            examples
            .sample(min(sampl_n, len(examples)))
            .to_string(index=False, header=False)
        )


    # Print longest example (if text area)
    print("\nLongest non-missing examples:")
    print("-" * 20)
    if len(examples) == 0:
        print("No non-missing examples available.")
    else:
        longest_examples = (
            examples
            .assign(length=examples[col_name].astype(str).str.len())   # creates new length column for sorting
            .sort_values("length", ascending=False)
            .drop(columns="length")                                    # drop length column on display                   
            .head(sampl_n)
        )
        print(longest_examples.to_string(index=False, header=False))

    
    # Print shortest example (sneaky NaNs)
    print("\nShortest non-missing examples:")
    print("-" * 20)
    if len(examples) == 0:
        print("No non-missing examples available.")
    else:
        shortest_examples = (
            examples
            .assign(length=examples[col_name].astype(str).str.len())
            .sort_values("length", ascending=True)
            .drop(columns="length")
            .head(sampl_n)
        )
        print(shortest_examples.to_string(index=False, header=False))


In [5]:
# Other Basic Tools
# df.shape, df.head()
# df.columns
# df.dtypes, df.info()
# df.isna().sum(), df.count().sort_values(ascending=True)
# df.nunique(), df["some_column"].value_counts(dropna=False)

In [6]:
raw_sample.head()

,Last Updated,Unique ID,OFSI Group ID,UN Reference Number,Name 6,Name 1,Name 2,Name 3,Name 4,Name 5,...,IMO number,Current owner/operator (s),Previous owner/operator (s),Current believed flag of ship,Previous flags,Type of ship,Tonnage of ship,Length of ship,Year Built,Hull identification number (HIN)
0,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
raw_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 57033 entries, 0 to 57032
Data columns (total 58 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Last Updated                                57033 non-null  str    
 1   Unique ID                                   57033 non-null  str    
 2   OFSI Group ID                               54769 non-null  float64
 3   UN Reference Number                         20532 non-null  str    
 4   Name 6                                      56921 non-null  str    
 5   Name 1                                      23893 non-null  str    
 6   Name 2                                      12107 non-null  str    
 7   Name 3                                      2573 non-null   str    
 8   Name 4                                      477 non-null    str    
 9   Name 5                                      101 non-null    str    
 10  Name type            

In [8]:
profile_column(raw_sample, "Subsidiaries", top_n=10)

########################################
Column: Subsidiaries
########################################
Total rows:          57033
Data type:           str
Non-missing values:  5595
Missing values:      51438
Missing %:           90.19%
Unique values:       206
Most common value:   AIS Iran Co
Most common count:   420

Top value counts:
--------------------
Subsidiaries
NaN                                                51438
AIS Iran Co                                          420
Electronic Component Industries (ECI)                420
Iranian Electronic Science & Research Institute      420
Iran Electronics Industries Co (Saga)                420
Isfahan Optics Industry (SAPA)                       420
Security Industry Information Space (SASTOBA)        420
Shiraz Electronics Industries (Sara Shiraz)          420
Telecommunication Industries of Iran (SAMA)          420
The Institute of Isayran Co                          420
Name: count, dtype: int64

Random non-missing examples:
--

In [9]:
"""
Other experiments
"""

# Checking pipelining in Subsidiaries and Parent company
# filt = raw_sample[
#     (raw_sample["Subsidiaries"].str.contains(";"))
# ]
# filt["Subsidiaries"]


# Checking entity specific information, ie. ships don't have passport number etc...
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Phone number", "Website", 
#               "Email address", "National Identifier number", 
#               "Passport number", "Business registration number (s)", "IMO number"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# Checking only individuals have DOB and Gender
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["D.O.B", "Gender"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# Checking Country information per entity type
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Nationality(/ies)", "Country of birth", "Current believed flag of ship", "Previous flags"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")
    
# Making sure only entities have subsidiaries or parent companies
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Subsidiaries", "Parent company"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")

# Making sure only entities have subsidiaries or parent companies
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Title"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# Seeing name relationships across entities (drop these later for space)
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Name type", "Alias strength", "Name non-latin script", "Non-latin script type", "Non-latin script language"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# filt = raw_sample[
#     (raw_sample["Designation Type"] == "Ship") &
#     (raw_sample["Name 6"].notna())
# ]
# filt


# Check the relationship between designation type and type of entity (does it apply to businesses only?)
# filt = raw_sample[
#     (raw_sample["Designation Type"] == "Entity") &
#     (raw_sample["Type of entity"].notna())
# ]
# len(filt)



# Seeing how many NaNs for alias strength there are even if the name type is an alias
# filt = raw_sample[
#     ((raw_sample["Name type"] == "Alias") | (raw_sample["Name type"] == "ALias")) &
#     (raw_sample["Alias strength"].notna())
# ]
# len(filt)


# filt = raw_sample[
#     (raw_sample["Name type"].isna())
# ]
# filt.iloc[0]


# Date Designated (most recent)
# pd.to_datetime(raw_sample["Date Designated"]).max()


# Address Format checking
# filt = raw_sample[
#     (raw_sample["Address Line 1"].str.contains("TSUEN WAN, NEW TERRITORIES, HONG KONG,UNIT 601", na=False))
# ]
# filt.iloc[0]

# Country Info Checking
# filt = raw_sample[
#     (raw_sample["Town of birth"].notna()) &
#     (raw_sample["Country of birth"].isna()) &
#     (raw_sample["Address Country"].isna()) &
#     (raw_sample["Nationality(/ies)"].isna())
# ]

# ID Differentiation
# filt = raw_sample[
#     (raw_sample["UN Reference Number"] == "TAi.004") 
# ]
# filt["Unique ID"].value_counts()

# Spotting Associations across fields
# filt[["Regime Name", "Designation Type", "Designation source", "Address Country", "Nationality(/ies)", "Country of birth"]].sample(n=20, replace=True)# ["Regime Name"].iloc[0]

# Non-latin script type and Non-latin script language
# filt = raw_sample[
#     (raw_sample["Name non-latin script"].notna()) &
#     (raw_sample["Non-latin script type"].isna()) 
# ]
# filt.iloc[0]


# Checking Name poor formatting
# filt = raw_sample[
#     (raw_sample["Name non-latin script"].notna()) &
#     (raw_sample["Name 1"].isna()) &
#     (raw_sample["Name 6"].isna())
# ]
# filt.iloc[0]


'\nOther experiments\n'

### Findings

Guiding the data cleaning process. Metadata found here: https://www.gov.uk/guidance/format-guide-for-the-uk-sanctions-list

The relevant columns that will contribute to building the transformed data source directly:
   - <span style="color: #61E283;">**Unique ID**</span>: Leaving this in as a reference/foreign key to the original sanctions dataset could help validate their presence in the original dataset later on. Not useful for matching, but has no missing values and could help keep a relationship with our transformed dataset.
   - <span style="color: #61E283;">**Name 1-6**</span>: Very important name information. Name 6 is the surname or the full name of the entity/ship. Names 1-5 are first and middle names, with increasing missing values as expected for rarer longer names
   - <span style="color: #61E283;">**Name type**</span>: Potentially helpful information if aliases are matched with innocent parties, this can be used with alias strength to determine a matching confidence score
   - <span style="color: #61E283;">**Alias strength**</span>: as above
   - <span style="color: #61E283;">**Name non-latin script**</span>: certainly useful for those few percent that may write their names in a different alphabet, helps make the matching system more robust
   - <span style="color: #61E283;">**Regime Name**</span>: not useful for matching but gives the bank context on which sanctions regime applies to the entity, it is also dense with no missing values
   - <span style="color: #61E283;">**Designation Type**</span>: provides context about the name, are they an individual, entity or ship (not directly useful for matching but can contribute to match validation/confidence)
   - <span style="color: #61E283;">**Sanctions Imposed**</span>: provides context about the type of sanctions the matched entity may face (again, not useful directly for matching, but may guide the bank's procedure after matching)
   - <span style="color: #61E283;">**Other Information**</span>: provides context about the person and their sanctions which isn't useful directly for matching, but can help inform later decisions. It may also be used to determine associated countries
   - <span style="color: #61E283;">**UK Statement of Reasons**</span>: as above, provides more context that could be used to further clarify situation after matching, not necessarily useful for the actual matching
   - <span style="color: #61E283;">**Type of entity**</span>: as above, provides more context that could be used to further clarify situation after matching, not necessarily useful for the actual matching (but for business/entity organisations rather than individuals)
   - <span style="color: #61E283;">**Address 1-6, Postal Code, Country**</span>: could use the address to match customer records or further improve matching confidence score. (also find associated countries)
   - <span style="color: #61E283;">**Phone number, Website, Email address, National Identifier number, Passport number, Business registration number, IMO number**</span>: further information that can guide matching confidence or be used as secondary matching criteria
   - <span style="color: #61E283;">**D.O.B, Gender**</span>: futher validation for name matching (eg. two people with same name but different birthdays, one is innocent)
   - <span style="color: #61E283;">**Nationality(/ies), Country of birth, Current believed flag of ship, previous flags**</span>: useful associated countries information
   - <span style="color: #61E283;">**Subsidiaries, Parent Company**</span>: the matching system likely needs to flag these too, treated as separate entity names for example (eg. companies under parent company being sanctioned)
   - <span style="color: #61E283;">**Current owner, Previous owner**</span>: similar to above but for ships
   - <span style="color: #61E283;">**Type of ship, Tonnage of ship, Length, Year built**</span>: validate match confidence but for ships



The columns we decided have no use for us are:
   - <span style="color: #F76262;">**Last Updated**</span>: If an entity already appears in this list, we would want the system to match them. There is no date of release of sanction that could be used alongside this field to clarify the sanctions validity.
   - <span style="color: #F76262;">**Date Designated**</span>: for similar reasons to the above
   - <span style="color: #F76262;">**OFSI Group ID**</span>: It was thought this could be used as a compound key with the Unique ID, but after exploring further, it held no differentiating power (legacy ID)
   - <span style="color: #F76262;">**UN Reference Number**</span>: similar to the above, it was mostly missing and held no differentiating power
   - <span style="color: #F76262;">**Designation source**</span>: it is already assumed we are curating a dataset for UK sanctions, and this field only differentiates between UN or UK which both matter (redundant)
   - <span style="color: #F76262;">**HIN**</span>: NaN column, can be discarded
   

These fields may be helpful, for example for imputation, but will ultimately not be needed in our transformed dataset:
   - <span style="color: #d48748;">**Title**</span>: some titles can be very distinguishing (Second Vice-President of the National Consti...), others may be very general (Captain, General...), shouldn't reliably be used for matching (or even match checking), titles may change, its mostly missing values, but maybe can impute country information or things like that??
   - <span style="color: #d48748;">**Position**</span>: similar to above
   - <span style="color: #d48748;">**Non-latin script type**</span>: not helpful for name matching purposes, but may help discern country information??
   - <span style="color: #d48748;">**Non-latin script language**</span>: as above
   - <span style="color: #d48748;">**National Identifier additional information, Passport additional information**</span>: not useful for matching, does provide context but not highly relevant. it may be used for identifying associated countries from the text though.
   - <span style="color: #d48748;">**Town of birth**</span>: can be used to impute country association data

## Designing
This is where we can start to formulate what our database may look like, consolidating our exploration ideas before data cleaning/transformation ensues

The proposed solution involves cleaning and normalising the dataset into 4 component datasets:
1. <span style="color: #61E283;">**name_index.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: newly created Sanctioned Entity ID, this can function as our new primary key, and foreign key across the datasets
    - <span style="color: #5d8eb8;">Full Name</span>: full name field, including any middle names and surname
    - <span style="color: #5d8eb8;">Designation Type</span>: identifies whether the party is an individual, entity or a ship
2. <span style="color: #61E283;">**individuals.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">LUID</span>: Legacy Unique ID, keeps the relationship to the original gov.uk dataset
    - <span style="color: #5d8eb8;">Surname</span>: from Name 6
    - <span style="color: #5d8eb8;">Given Names</span>: Name 1+2+...+5
    - <span style="color: #5d8eb8;">Name (Non-Latin Script)</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Type</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Language</span>: =
    - <span style="color: #5d8eb8;">Name Type</span>: =
    - <span style="color: #5d8eb8;">Alias Strength</span>: =
    - <span style="color: #5d8eb8;">D.O.B</span>: =
    - <span style="color: #5d8eb8;">Gender</span>: =
    - <span style="color: #5d8eb8;">Title</span>: =
    - <span style="color: #5d8eb8;">Position</span>: =
    - <span style="color: #5d8eb8;">Nationalities</span>: =
    - <span style="color: #5d8eb8;">Birth Country</span>: =
    - <span style="color: #5d8eb8;">Birth Town</span>: =
    - <span style="color: #5d8eb8;">Address Lines</span>: Address Lines 1+2+...+6
    - <span style="color: #5d8eb8;">Address Postal Code</span>: =
    - <span style="color: #5d8eb8;">Address Country</span>: =
    - <span style="color: #5d8eb8;">Phone Number</span>: =
    - <span style="color: #5d8eb8;">Website</span>: =
    - <span style="color: #5d8eb8;">Email</span>: =
    - <span style="color: #5d8eb8;">National Identifier Number</span>: =
    - <span style="color: #5d8eb8;">National Identifier Info</span>: =
    - <span style="color: #5d8eb8;">Passport Number</span>: =
    - <span style="color: #5d8eb8;">Passport Info</span>: =
    - <span style="color: #5d8eb8;">Other Info</span>: =
    - <span style="color: #5d8eb8;">UK Statement of Reasons</span>: =

3. <span style="color: #61E283;">**entities.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">LUID</span>: ^^
    - <span style="color: #5d8eb8;">Name</span>: Usually just Name 6, but concat 1-6
    - <span style="color: #5d8eb8;">Name (Non-Latin Script)</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Type</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Language</span>: =
    - <span style="color: #5d8eb8;">Name Type</span>: =
    - <span style="color: #5d8eb8;">Alias Strength</span>: =
    - <span style="color: #5d8eb8;">Address Lines</span>: Address Lines 1+2+...+6
    - <span style="color: #5d8eb8;">Address Postal Code</span>: =
    - <span style="color: #5d8eb8;">Address Country</span>: =
    - <span style="color: #5d8eb8;">Phone Number</span>: =
    - <span style="color: #5d8eb8;">Website</span>: =
    - <span style="color: #5d8eb8;">Email</span>: =
    - <span style="color: #5d8eb8;">Business Reg</span>: =
    - <span style="color: #5d8eb8;">Type</span>: from type of entity
    - <span style="color: #5d8eb8;">Subsidiaries</span>: =
    - <span style="color: #5d8eb8;">Parent Company</span>: =
    - <span style="color: #5d8eb8;">Other Info</span>: =
    - <span style="color: #5d8eb8;">UK Statement of Reasons</span>: =
    
4. <span style="color: #61E283;">**ships.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">LUID</span>: ^^
    - <span style="color: #5d8eb8;">Name</span>: Usually just Name 6, but concat 1-6
    - <span style="color: #5d8eb8;">Name (Non-Latin Script)</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Type</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Language</span>: =
    - <span style="color: #5d8eb8;">Name Type</span>: =
    - <span style="color: #5d8eb8;">Alias Strength</span>: =
    - <span style="color: #5d8eb8;">IMO Number</span>: =
    - <span style="color: #5d8eb8;">Current Believed Flag</span>: =
    - <span style="color: #5d8eb8;">Previous Flags</span>: =
    - <span style="color: #5d8eb8;">Type</span>: from type of ship
    - <span style="color: #5d8eb8;">Tonnage</span>: =
    - <span style="color: #5d8eb8;">Length</span>: =
    - <span style="color: #5d8eb8;">Year Built</span>: =
    - <span style="color: #5d8eb8;">Other Info</span>: =
    - <span style="color: #5d8eb8;">UK Statement of Reasons</span>: =


5. <span style="color: #61E283;">**sanctions.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">Regime Name</span>: =
    - <span style="color: #5d8eb8;">Sanctions Imposed</span>: depipeline this ideally
    - <span style="color: #5d8eb8;">Date Designated</span>: =
    - <span style="color: #5d8eb8;">Last Updated</span>: =

The motivating idea behind this scheme is that it is very likely that the customer records will include a name or a designation type. Here is the typical algorithm flow:

1. Designation Type is known
    - Perform the matching algorithm on the narrower datasets (either: individuals, entities or ships .csv) with whatever fields you have

2. Otherwise, Name is known
    - Use name_index.csv to find candidate SEID foreign keys and their designation types, and check those corresponding csv and keys further for matching validity/confidence etc...
    - Datasets can be sorted by SEID to allow for binary search

3. Otherwise, Designation Type and Name is not known (e.g. only have an address)
    - Look at the csv headers for clues about the designation type (e.g. only ships have IMO numbers, only individuals have Passport numbers, etc...)
    - Otherwise if limited matching criteria to go off of, scan for matches across all 3 datasets (worst case scenario)

Note: name_index could be expanded with more fields, but there aren't other useful shared fields between all three entity types


## Transform

Create the first instances of each dataset in our database (clean and deduplicate them later)

In [10]:
# Remove unused/unneeded columns
not_used = [
    "OFSI Group ID",
    "UN Reference Number",
    "Designation source",
    "Hull identification number (HIN)"
]
raw_sample.drop(not_used, axis=1, inplace=True)

### 1. name_index.csv

Observations:\
    - Lots of NaNs from later Name fields (longer names less likely)\
    - Some individuals have their full name written in name 1, rather than formatting the surname in name 6\
    - Some people only have a non-latin script name, and vice versa\
    - Varying case script, snakescript etc...\
    - 3 rare entity records had a name 1 field too, not just name 6

i. Create an SEID for each entry in the full dataset

In [11]:
raw_sample["SEID"] = [
    f"#{str(i).zfill(7)}" for i in range(len(raw_sample))
]

ii. Create a Full Name field from merging names 1-6 in order\
The matching algorithm can look for substring similarities within one more densely populated names field, rather than 5 sparse ones

In [12]:
raw_sample["Full Name"] = (
    raw_sample[[                              # Ordered Name columns 1-6
        "Name 1",
        "Name 2",
        "Name 3",
        "Name 4",
        "Name 5",
        "Name 6"
    ]]
    .fillna("")                               # Replace NaN with empty string
    .agg(" ".join, axis=1)                    # Joins name fields with space in between (across columns)
    .str.replace(r"\s+", " ", regex=True)     # Cleans up extra whitespace characters (replace them with just 1 space)
    .str.strip()                              # Remove any left/right trailing spaces
)

# Initialise name_index dataframe
name_index = raw_sample[["SEID", "Full Name", "Designation Type"]].copy()
# raw_sample.drop("Full Name", axis=1, inplace=True)

iii. Extract any other names from the records that may be helpful for matching. Keep the same SEIDS to preserve link to stable record in narrower dataset\
Fields of Interest: Name non-latin script, Subsidiaries, Parent Company, Current & Previous owner

In [13]:
# Other fields that may contain useful matchable names
name_sources = [
    "Name non-latin script",
    "Subsidiaries", 
    "Parent company",      
    "Current owner/operator (s)",
    "Previous owner/operator (s)"
]

# Container for extra rows
extra_names = []


for c in name_sources:
    # Keep the column info and the same SEID for consistency
    temp = raw_sample[raw_sample[c].notna()][["SEID", c, "Designation Type"]].copy()

    # Rename, ready to concat series form
    temp = temp.rename(columns={c: "Full Name"})
    extra_names.append(temp)
extra_names = pd.concat(extra_names, ignore_index=True)

# Add the new names to name_index
name_index = pd.concat(
    [name_index, extra_names],
    ignore_index=True
)

iv. Explode any multiple values per cell, using classic delimiters like ; or | or \n that are unlikely to be naturally occuring in text as parts of names

In [14]:
# Split text whenever delimiter appears with 0+ spaces either side (formatting)
name_index["Full Name"] = (
    name_index["Full Name"]
    .str.split(r"\s*[;|\n]\s*", regex=True)   
)

# List to explode across rows
name_index = name_index.explode("Full Name", ignore_index=True)

# Reformatting (if original text was badly spaced)
name_index["Full Name"] = (
    name_index["Full Name"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [15]:
name_index[name_index["SEID"] == "#0045434"]

,SEID,Full Name,Designation Type
45434,#0045434,PUBLIC JOINT STOCK COMPANY SOVCOMFLOT,Entity
61681,#0045434,Публичное акционерное общество Современный ком...,Entity
69465,#0045434,OOO SCF Arctic,Entity
69466,#0045434,SCF Management Services (Cyprus) Ltd,Entity
69467,#0045434,PAO Novoship,Entity
69468,#0045434,SCF management Services (St. Petersburg) Ltd,Entity
69469,#0045434,Sovcomflot (UK) Ltd,Entity
69470,#0045434,SCF Management Services (St. Petersburg) Ltd s...,Entity
69471,#0045434,,Entity


In [16]:
raw_sample[raw_sample["SEID"] == "#0045434"][["Name 6", "Subsidiaries", "Parent company", "Name non-latin script"]]

,Name 6,Subsidiaries,Parent company,Name non-latin script
45434,PUBLIC JOINT STOCK COMPANY SOVCOMFLOT,OOO SCF Arctic; SCF Management Services (Cypru...,NaN,Публичное акционерное общество Современный ком...


In [17]:
# Save and delete for memory
name_index.to_csv("name_index.csv", index=False)
del name_index
gc.collect()

0

### 2. individuals.csv

i. Merge first and middle names into a Given Names field

In [18]:
raw_sample["Given Names"] = (
    raw_sample[[                              # Ordered Name columns 1-6
        "Name 1",
        "Name 2",
        "Name 3",
        "Name 4",
        "Name 5"
    ]]
    .fillna("")                               # Replace NaN with empty string
    .agg(" ".join, axis=1)                    # Joins name fields with space in between (across columns)
    .str.replace(r"\s+", " ", regex=True)     # Cleans up extra whitespace characters (replace them with just 1 space)
    .str.strip()                              # Remove any left/right trailing spaces
)

ii. Merge address 1-6 into an Address Lines field\
Similar searching logic, here a multi-value cell delimited by ; makes sense from a storage point of view

In [19]:
raw_sample["Address Lines"] = (
    raw_sample[[                              # Ordered Name columns 1-6
        "Address Line 1",
        "Address Line 2",
        "Address Line 3",
        "Address Line 4",
        "Address Line 5",
        "Address Line 6",
    ]]
    .fillna("")                               # Replace NaN with empty string
    .agg("; ".join, axis=1)                   # Joins name fields with ; in between (across columns) (multi value cell)
    .str.replace(r"\s+", " ", regex=True)     # Cleans up extra whitespace characters (replace them with just 1 space)
    .str.strip()                              # Remove any left/right trailing spaces
)

iii. Initialise the individuals.csv dataset

In [20]:
# Initialise the individuals.csv
individuals = raw_sample[raw_sample["Designation Type"] == "Individual"][[
    "SEID",
    "Unique ID",
    "Name 6",
    "Given Names",
    "Name non-latin script",
    "Non-latin script type",
    "Non-latin script language",
    "Name type",
    "Alias strength",
    "D.O.B",
    "Gender",
    "Title",
    "Position",
    "Nationality(/ies)",
    "Country of birth",
    "Town of birth",
    "Address Lines",
    "Address Postal Code",
    "Address Country",
    "Phone number",
    "Website",
    "Email address",
    "National Identifier number",
    "National Identifier additional information",
    "Passport number",
    "Passport additional information",
    "Other Information",
    "UK Statement of Reasons"
]].copy()

iv. Rename columns, save and delete for memory

In [21]:
rename_map = {
    "Unique ID": "LUID",
    "Name 6": "Surname",
    "Name non-latin script": "Name (Non-Latin Script)",
    "Non-latin script type": "Non-Latin Script Type",
    "Non-latin script language": "Non-Latin Script Language",
    "Name type": "Name Type",
    "Alias strength": "Alias Strength",
    "Nationality(/ies)": "Nationalities",
    "Country of birth": "Birth Country",
    "Town of birth": "Birth Town",
    "Phone number": "Phone Number",
    "Email address": "Email",
    "National Identifier number": "National Identifier Number",
    "National Identifier additional information": "National Identifier Info",
    "Passport number": "Passport Number",
    "Passport additional information": "Passport Info"
}

individuals = individuals.rename(columns=rename_map)

In [22]:
# Save and delete for memory
individuals.to_csv("individuals.csv", index=False)
del individuals
gc.collect()

0

### 3. entities.csv

i. Initialise the entities.csv dataset

In [23]:
entities = raw_sample[raw_sample["Designation Type"] == "Entity"][[
    "SEID",
    "Unique ID",
    "Full Name",
    "Name non-latin script",
    "Non-latin script type",
    "Non-latin script language",
    "Name type",
    "Alias strength",
    "Address Lines",
    "Address Postal Code",
    "Address Country",
    "Phone number",
    "Website",
    "Email address",
    "Business registration number (s)",
    "Type of entity",
    "Subsidiaries",
    "Parent company",
    "Other Information",
    "UK Statement of Reasons"
]].copy()

ii. Rename columns, save and delete for memory

In [24]:
rename_map = {
    "Unique ID": "LUID",
    "Name non-latin script": "Name (Non-Latin Script)",
    "Non-latin script type": "Non-Latin Script Type",
    "Non-latin script language": "Non-Latin Script Language",
    "Name type": "Name Type",
    "Alias strength": "Alias Strength",
    "Phone number": "Phone Number",
    "Email address": "Email",
    "Business registration number (s)": "Business Reg",
    "Type of entity": "Type",
    "Parent company": "Parent Company"
}

entities = entities.rename(columns=rename_map)

In [25]:
#entities.info()

In [26]:
#raw_sample["Designation Type"].value_counts()

In [27]:
# Save and delete for memory
entities.to_csv("entities.csv", index=False)
del entities
gc.collect()

0

### 4. ships.csv

i. Initialise the ships.csv dataset

In [28]:
ships = raw_sample[raw_sample["Designation Type"] == "Ship"][[
    "SEID",
    "Unique ID",
    "Full Name",
    "Name non-latin script",
    "Non-latin script type",
    "Non-latin script language",
    "Name type",
    "Alias strength",
    "IMO number",
    "Current believed flag of ship",
    "Previous flags",
    "Type of ship",
    "Tonnage of ship",
    "Length of ship",
    "Year Built",
    "Other Information",
    "UK Statement of Reasons"
]].copy()

ii. Rename columns, save and delete for memory

In [29]:
rename_map = {
    "Unique ID": "LUID",
    "Name non-latin script": "Name (Non-Latin Script)",
    "Non-latin script type": "Non-Latin Script Type",
    "Non-latin script language": "Non-Latin Script Language",
    "Name type": "Name Type",
    "Alias strength": "Alias Strength",
    "IMO number": "IMO Number",
    "Current believed flag of ship": "Current Believed Flag",
    "Previous flags": "Previous Flags",
    "Type of ship": "Type",
    "Tonnage of ship": "Tonnage",
    "Length of ship": "Length"
}

ships = ships.rename(columns=rename_map)

In [30]:
#ships.info()

In [31]:
#raw_sample["Designation Type"].value_counts()

In [32]:
# Save and delete for memory
ships.to_csv("ships.csv", index=False)
del ships
gc.collect()

0

### 5. sanctions.csv

i. Initialise the sanctions.csv dataset

In [33]:
sanctions = raw_sample[[
    "SEID",
    "Regime Name",
    "Sanctions Imposed",
    "Date Designated",
    "Last Updated"
]].copy()

ii. Save and delete for memory

In [34]:
#sanctions.info()

In [35]:
#raw_sample["Designation Type"].value_counts()

In [36]:
# Save and delete for memory
sanctions.to_csv("sanctions.csv", index=False)
del sanctions
gc.collect()

0

## Cleaning

Clean each field in each of our datasets

Track the SEIDs of records that we can safely remove across all datasets

In [37]:
rm_seids = pd.Series([])

Simple whitespace formatting function

In [38]:
def whitespace_fmt(v):
    if pd.isna(v):
        return v                   # Leave NaNs untouched
    v = str(v)
    v = re.sub(r"\s+", " ", v)     # Only need one space at a time
    v = v.strip()                  # Left and Right truncate spaces
    return v

### 1. name_index.csv

In [39]:
name_index = pd.read_csv("name_index.csv")
name_index.info()

<class 'pandas.DataFrame'>
RangeIndex: 79798 entries, 0 to 79797
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   SEID              79798 non-null  str  
 1   Full Name         79465 non-null  str  
 2   Designation Type  79798 non-null  str  
dtypes: str(3)
memory usage: 1.8 MB


#### Process each field

##### - <span style="color: #5d8eb8;">SEID</span>:
- No Processing Required (Clean)

##### - <span style="color: #5d8eb8;">Full Name</span>:
- Some Missing values can be ammended if there are other name records with the same SEID
- There were 40 remaining NaN Name records. After inspecting them, they all had the same title (Haji), from kabul afghanistan, no name fields anywhere, incosistent (and incomplete) birth dates, the same passport number though, and the only available information was copied across all the records, reading: "A close associate of Mullah Mohammed Omar (TAi.004). Member of Taliban Supreme Council as at Dec. 2009. Belongs to Baabar tribe. Review pursuant to Security Council resolution 1822 (2008) was concluded on 21 Jul. 2010. INTERPOL-UN Security Council Special Notice web link: https://www.interpol.int/en/How-we-work/ Notices/View-UN-Notices-Individuals click here"
    - We decided it was safe to remove these records as mostly missing/incomplete data
- Some of the names seem bizzarely short, could cause false positive matches, but they could very well be valid aliases, and sometimes valid acronyms, like CP for Cyber Police

In [40]:
mssg = name_index[name_index["Full Name"].isna()]["SEID"].drop_duplicates()
removable = name_index[
    (name_index["SEID"].isin(mssg)) &
    (name_index["Full Name"].notna())
]["SEID"].drop_duplicates()

In [41]:
name_index[name_index["SEID"] == "#0022537"]

,SEID,Full Name,Designation Type
22537,#0022537,Myanmar Economic Corporation,Entity
64944,#0022537,Dagon FC Company Ltd.,Entity
64945,#0022537,NaN,Entity


In [42]:
name_index = name_index[~(
    (name_index["SEID"].isin(removable)) &
    (name_index["Full Name"].isna())
)]
del mssg
del removable
gc.collect()

0

In [43]:
still_mssg = name_index[name_index["Full Name"].isna()]["SEID"].drop_duplicates()
#name_index[name_index["Full Name"].isna()][["SEID", "Designation Type"]]

In [44]:
#raw_sample.columns

In [45]:
raw_sample[raw_sample["SEID"].isin(still_mssg)].iloc[:, :7]

,Last Updated,Unique ID,Name 6,Name 1,Name 2,Name 3,Name 4
1021,14/04/2026,AFG0006,NaN,NaN,NaN,NaN,NaN
1022,14/04/2026,AFG0006,NaN,NaN,NaN,NaN,NaN
1023,14/04/2026,AFG0006,NaN,NaN,NaN,NaN,NaN
1024,14/04/2026,AFG0006,NaN,NaN,NaN,NaN,NaN
1025,14/04/2026,AFG0006,NaN,NaN,NaN,NaN,NaN
1026,14/04/2026,AFG0006,NaN,NaN,NaN,NaN,NaN
1027,14/04/2026,AFG0006,NaN,NaN,NaN,NaN,NaN
1028,14/04/2026,AFG0006,NaN,NaN,NaN,NaN,NaN
1029,14/04/2026,AFG0006,NaN,NaN,NaN,NaN,NaN
1030,14/04/2026,AFG0006,NaN,NaN,NaN,NaN,NaN


In [46]:
# Track the SEID of these safe to remove missing records
rm_seids = pd.concat([rm_seids, still_mssg]).drop_duplicates()

In [47]:
#profile_column(name_index, "Full Name", key=["SEID"])

In [48]:
#raw_sample[raw_sample["SEID"] == "#0040983"].iloc[:,:]

In [49]:
name_index["Full Name"] = name_index["Full Name"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Designation Type</span>:
 - Already clean, 3 distinct clean value types

In [50]:
name_index["Designation Type"].value_counts()

Designation Type
Entity        46335
Individual    31539
Ship           1631
Name: count, dtype: int64

#### Save & Clear Memory

In [51]:
# Save and delete for memory
name_index.to_csv("name_index.csv", index=False)
del name_index
gc.collect()

0

### 2. individuals.csv

In [52]:
individuals = pd.read_csv("individuals.csv")
individuals.info()

<class 'pandas.DataFrame'>
RangeIndex: 26414 entries, 0 to 26413
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   SEID                        26414 non-null  str  
 1   LUID                        26414 non-null  str  
 2   Surname                     26312 non-null  str  
 3   Given Names                 23898 non-null  str  
 4   Name (Non-Latin Script)     5182 non-null   str  
 5   Non-Latin Script Type       2015 non-null   str  
 6   Non-Latin Script Language   1496 non-null   str  
 7   Name Type                   26356 non-null  str  
 8   Alias Strength              12982 non-null  str  
 9   D.O.B                       24809 non-null  str  
 10  Gender                      10642 non-null  str  
 11  Title                       2456 non-null   str  
 12  Position                    15117 non-null  str  
 13  Nationalities               21797 non-null  str  
 14  Birth Country    

#### Process each field

##### - <span style="color: #5d8eb8;">SEID, LUID</span>:
- No Processing Required (Clean)

In [53]:
#profile_column(individuals, "LUID")

In [54]:
individuals["LUID"] = individuals["LUID"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Surname, Given Names, Name (Non-Latin Script)</span>:
- Some names seem surprisingly short (e.g. surname, K), but these seem to be aliases/primary name variations that are valid
- There are many missing Given Names or Non-Latin Names, but at least those records have one of these other three name fields given, which is enough name information disqualifying them from removal.
- The only other missing names come from the already tracked nameless Haji rows

In [55]:
#profile_column(individuals, "Surname", key=["SEID"])
#individuals[individuals["SEID"] == "#0019689"]

In [56]:
# individuals[
#     (individuals["Surname"].isna()) &
#     (individuals["Given Names"].isna()) &
#     (individuals["Name (Non-Latin Script)"].isna())
# ]["SEID"].isin(rm_seids)7
# individuals[
#     (individuals["Given Names"].isna()) &
#     (individuals["Surname"].isna()) &
#     (individuals["Name (Non-Latin Script)"].isna())
# ]["SEID"].isin(rm_seids)
# individuals[
#     (individuals["Name (Non-Latin Script)"].isna()) &
#     (individuals["Surname"].isna()) &
#     (individuals["Given Names"].isna())
# ]["SEID"].isin(rm_seids)

In [57]:
individuals["Surname"] = individuals["Surname"].apply(whitespace_fmt)
individuals["Given Names"] = individuals["Given Names"].apply(whitespace_fmt)
individuals["Name (Non-Latin Script)"] = individuals["Name (Non-Latin Script)"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Non-Latin Script Type, Non-Latin Script Language</span>:
- Many missing values even in fields with a given Non-Latin Script Name
- We can try to impute some of the script types using regex
    - Note that this introduces script type Han, since it is not always clear if it is Chinese or Japanese script etc...
- However, it's difficult to discern script language from just a name field, we would have to use other information like birth country or address or nationalities, etc... (for example, cyrillic script is used in serbian, russian, etc...)
    - We could try do this if we had more time using an NLP like approach (LLM prompting)

In [58]:
# individuals[
#     (individuals["Given Names"] == "MOHAMMAD HASSAN") &
#     (individuals["Non-Latin Script Type"].notna())
# ]

In [59]:
# Detect the script from a given non-latin name
def detect_script(txt):
    txt = str(txt)
    for script in ["Arabic", "Cyrillic", "Han", "Khmer", 
                   "Georgian", "Hebrew", "Hangul", "Greek"]:
        if re.search(fr"\p{{{script}}}", txt):
            return script
    return None

In [60]:
impute_seid = individuals[ 
    (individuals["Non-Latin Script Type"].isna()) & 
    (individuals["Name (Non-Latin Script)"].notna()) 
]["SEID"]
mask = individuals["SEID"].isin(impute_seid)

In [61]:
individuals[individuals["SEID"].isin(impute_seid)][["Name (Non-Latin Script)", "Non-Latin Script Type"]]

,Name (Non-Latin Script),Non-Latin Script Type
40,محمد حسن آخوند,NaN
41,محمد حسن آخوند,NaN
42,محمد حسن آخوند,NaN
43,محمد حسن آخوند,NaN
44,محمد حسن آخوند,NaN
...,...,...
26343,صالح مسفر صالح الشاعر,NaN
26344,صالح مسفر صالح الشاعر,NaN
26345,صالح مسفر صالح الشاعر,NaN
26346,صالح مسفر صالح الشاعر,NaN


In [62]:
individuals.loc[mask, "Non-Latin Script Type"] = (
    individuals.loc[mask, "Name (Non-Latin Script)"]
    .apply(detect_script)
)
individuals[individuals["SEID"].isin(impute_seid)][["Name (Non-Latin Script)", "Non-Latin Script Type"]]

,Name (Non-Latin Script),Non-Latin Script Type
40,محمد حسن آخوند,Arabic
41,محمد حسن آخوند,Arabic
42,محمد حسن آخوند,Arabic
43,محمد حسن آخوند,Arabic
44,محمد حسن آخوند,Arabic
...,...,...
26343,صالح مسفر صالح الشاعر,Arabic
26344,صالح مسفر صالح الشاعر,Arabic
26345,صالح مسفر صالح الشاعر,Arabic
26346,صالح مسفر صالح الشاعر,Arabic


In [63]:
individuals[individuals["SEID"].isin(impute_seid)]["Non-Latin Script Type"].value_counts()

Non-Latin Script Type
Arabic      1445
Cyrillic    1207
Han            3
Greek          1
Name: count, dtype: int64

In [64]:
# individuals[individuals["SEID"].isin(impute_seid)&
#             (individuals["Non-Latin Script Type"].isna())]

In [65]:
# profile_column(individuals, "Non-Latin Script Type", top_n=12)

In [66]:
del impute_seid
del mask
gc.collect()

0

##### - <span style="color: #5d8eb8;">Name Type, Alias Strength</span>:
- Inconsistent values for the Name Type nominal field, eg. ALias, or Primary name variation, we can map these to a consistent value
    - But for the other missing values (excluding rm_seids), imputation still requires domain-sepcific knowledge as to whether its an Alias or Primary Name etc...
- Alias Strength can't be imputed for similar lack of domain-specific knowledge. At this stage, there were 733 given Alias Name Types without their strength recorded.

In [67]:
name_type_map = {
    "Primary name": "Primary Name",
    "Primary name variation": "Primary Name Variation",
    "ALias": "Alias"
}

individuals["Name Type"] = individuals["Name Type"].replace(name_type_map)

In [68]:
# individuals[
#     individuals["Name Type"].isna() &
#     ~(individuals["SEID"].isin(rm_seids))
# ]

In [69]:
#profile_column(individuals, "Name Type", top_n=10)

In [70]:
# individuals[
#     individuals["Alias Strength"].isna() &
#     (individuals["Name Type"] == "Alias") &
#     ~(individuals["SEID"].isin(rm_seids))
# ]

In [71]:
#profile_column(individuals, "Alias Strength", top_n=10)

##### - <span style="color: #5d8eb8;">D.O.B</span>:
- Not in date format due to unknown birth months or days for example: dd/mm/1955
- But some records give just the year 1971, when for matching and consistency, we should have dd/mm/1971
    - Also one date accidentally has an apostrophe: '00/00/1975
- Otherwise, we can't reliably impute the rest of the missing values

In [72]:
# Fix the random apostrophe
individuals.loc[individuals["SEID"] == "#0046566", "D.O.B"] = "00/00/1975"

In [73]:
# individuals[
#     (individuals["D.O.B"].str.len() != 10) &
#     (individuals["D.O.B"].notna())
# ]["D.O.B"][150:200]

# Fix formatting of dates consistently
impute_seid = individuals[
    (individuals["D.O.B"].str.len() != 10) &
    (individuals["D.O.B"].notna())
]["SEID"]
mask = individuals["SEID"].isin(impute_seid)
individuals.loc[mask, "D.O.B"] = (
    individuals.loc[mask, "D.O.B"]
    .apply(lambda x: "dd/mm/"+x)
)

In [74]:
#individuals[individuals["D.O.B"].notna() & (individuals["D.O.B"].str.len() != 10)]

In [75]:
#profile_column(individuals, "D.O.B", key=["SEID"])

##### - <span style="color: #5d8eb8;">Gender</span>:
- Inconsistent value types for this nominal field, eg. title vs lower case. We can fix this by mapping to title case consistently
- We can try use very obvious/common title information to impute gender too
- Otherwise, can't reliably impute the rest of the NaNs

In [76]:
gender_map = {
    "male": "Male",
    "female": "Female"
}

individuals["Gender"] = individuals["Gender"].replace(gender_map)

In [77]:
#individuals["Title"].value_counts().index
#profile_column(individuals, "Gender", top_n=10)

In [78]:
title_gender_map = {
    "Haji": "Male",
    "Mr": "Male",
    "Maulavi": "Male",
    "Sheikh": "Male",
    "Shaikh": "Male",
    "Hajji": "Male",
    "Qari": "Male",
    "Alhaj": "Male",
    "Ms": "Female",
    "Mawlawi": "Male",
    "Amir": "Male",
    "Ustad": "Male",
    "Imam": "Male",
    "Sheik": "Male",
    "Mrs": "Female",
    "Maulawi": "Male",
    "Ustadz": "Male",
    "Maulana": "Male",
    "Miss": "Female",
    "Mufti": "Male",
    "Sayyed": "Male"
}
mask = (
    individuals["Gender"].isna() &
    (individuals["Title"].notna())
)
individuals.loc[mask, "Gender"] = (
    individuals.loc[mask, "Title"]
    .map(title_gender_map)
)

In [79]:
#profile_column(individuals, "Gender", top_n=10)

##### - <span style="color: #5d8eb8;">Title</span>:
- Mostly missing field that can't be imputed, but in rare circumstances it may help distinguish individuals for the matching algorithm, I've decided to keep it but it could ultimately be dropped

In [80]:
#profile_column(individuals, "Title", top_n=10)

In [81]:
individuals["Title"] = individuals["Title"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Position</span>:
- 5 entries had a " " empty string position, so we converted it to NaN for consistency
- Also noticed a 'general', 'ormer Chief of Staff of the Sudan People’s Liberation Army (SPLA)', we can map these to what they should be for consistency

In [82]:
#individuals.loc[individuals["Position"] == " ", "Position"]
individuals.loc[individuals["Position"] == " ", "Position"] = None

In [83]:
#profile_column(individuals, "Position", top_n=10, key=["SEID"])
#individuals["Position"].value_counts().index.to_list()

In [84]:
position_map = {
    "'general'": "General",
    "ormer Chief of Staff of the Sudan People’s Liberation Army (SPLA)": "Former Chief of Staff of the Sudan People’s Liberation Army (SPLA)"
}

individuals["Position"] = individuals["Position"].replace(position_map)

In [85]:
#individuals["Position"].value_counts().index.to_list()

In [86]:
individuals["Position"] = individuals["Position"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Nationalities</span>:
- Formatting seemed in order here, but when it comes to imputing missing values, this may be unreliable from the rest of the fields since different nations have different rules.

In [87]:
#profile_column(individuals, "Nationalities", top_n=10, key=["SEID"])
#individuals["Nationalities"].value_counts().index.to_list()
individuals["Nationalities"] = individuals["Nationalities"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Birth Country, Birth Town</span>:
- There are some inconsistencies between birth country names, like "former USSR Currently Russia", "Russian SFSR, (now Russian Federation)", and "Russian Soviet Federative Socialist Republic (RSFSR)"... but it is unclear how best to consolidate these without losing potentially meaningful information.
- It may be possible to use geocoding libraries such as geopy to infer missing birth country values from birth town data. However, this introduces reliance on an external geocoding service, may be affected by rate limits, and can produce ambiguous results (eg. if town names are not unique).

In [88]:
#individuals["Birth Country"].value_counts().index.to_list()
#individuals["Birth Town"].value_counts().index.to_list()

In [89]:
# impute_seid = individuals[
#     (individuals["Birth Country"].isna()) &
#     (individuals["Birth Town"].notna())
# ]["SEID"]
# mask = individuals["SEID"].isin(impute_seid)

# individuals.loc[mask, "Birth Country"] = (
#     """Geopy Here"""
# )

In [90]:
#profile_column(individuals, "Birth Town", top_n=10, key=["SEID"])
#individuals["Birth Town"].value_counts().index.to_list()

In [91]:
individuals["Birth Country"] = individuals["Birth Country"].apply(whitespace_fmt)
individuals["Birth Town"] = individuals["Birth Town"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Address: Lines, Postal Code, Country</span>:
- Due to the nature of how I joined the address lines into one field, there are many empty ; ; ;... sequences, eg. NaNs are now ; ; ; ; ; and we have other examples like: Kabul; ; ; ; ; or ; ; ; ; Ndélé; Bamingui-Bangoran. We can clear any useless ; and bring back the NaNs
- For the Address Postal Codes, we can also make the PO Box values more consistent, so we dont have values like: PO Box, PO BOX or P.O.
    - but some miscellaneous values remain that don't have a clear processing route, eg: Kampong Speu 5301 seems to be a legitimate postal code in Cambodia, not a Town+Code formatting issue
    - Note: there was also one postal code with an apostrophe: '0578
- The Address Country field appeared to be in order, good formatting
- Imputing Postal Code and Country information from the Address Lines field may also be possible, but due to potential rate limiting from geoencoder libraries, we opt to not do this now (maybe in the future)

In [92]:
# Function to clean up the address lines field
def clean_address_lines(v):
    if pd.isna(v):
        return np.nan
    
    v = str(v)
    parts = v.split(";")
    parts = [part.strip() for part in parts if part.strip() != ""]

    # Check for ; ; ; ; Nan case
    if len(parts) == 0:
        return np.nan
    
    # Otherwise, rejoin the useful parts
    return " ; ".join(parts)

# Function to standardise variations of postal codes
def clean_postcode(v):
    if pd.isna(v):
        return np.nan

    # Standardise PO Box variations
    v = str(v).strip()
    v = re.sub(                      # Finds matches of re in v and substitutes it with "PO Box" standardised form
        r"\bP\.?\s*O\.?\s*BOX\b",    # Match P at start of word, 1 optional ., 0+ spaces, O, 1 optional ., 
        "PO Box",                    # 0+ spaces again, BOX with a word boundary so it ends there (and we set case insensitivity)
        v,
        flags=re.IGNORECASE
    )

    # Remove apostrophes and quotation characters
    v = re.sub(r"[\'’‘`´]", "", v)

    # Keep only letters, numbers, spaces, and hyphens
    v = re.sub(r"[^A-Za-z0-9\s-]", " ", v)
    # Note: [^] means match characters not in the set

    # Also standardise to upper case for now
    v = v.upper()

    # Set empty to NaN
    if v == "":
        return np.nan

    return v

In [93]:
#profile_column(individuals, "Address Lines", top_n=10, key=["SEID"])
#individuals["Address Postal Code"].value_counts().index.to_list()

In [94]:
individuals["Address Lines"] = individuals["Address Lines"].apply(clean_address_lines)
individuals["Address Postal Code"] = individuals["Address Postal Code"].apply(clean_postcode)
#individuals.loc[individuals["SEID"] == "#0010357", "Address Postal Code"] = "0578"


In [95]:
#profile_column(individuals, "Address Postal Code", top_n=10, key=["SEID"])
#individuals["Address Postal Code"].value_counts().index.to_list()
#len(individuals["Address Postal Code"].value_counts().index.to_list())

In [96]:
individuals["Address Lines"] = individuals["Address Lines"].apply(whitespace_fmt)
individuals["Address Postal Code"] = individuals["Address Postal Code"].apply(whitespace_fmt)
individuals["Address Country"] = individuals["Address Country"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Phone Number</span>:
- This field certainly has a lot of variation in formatting, adn there are cells like: 9562236 (08 Apr 2010 - ) 74955098211 (29 Aug 2007 - )  5098211 (14 Jun 2006 - ) which are hard to interpret, since perhaps those dates are meaningful and should not be removed via regex
- Additionally, matching algorithms for phone numbers can focus on the numeric match query and look for subsequence matching in the phone number field (skipping characters like () or -)
- Nonetheless, there were some whitespace formatting issues that can be fixed, eg: +79165359238\u202f, and we can remove unexpected characters like apostrophes

In [97]:
#profile_column(individuals, "Phone Number", top_n=10, key=["SEID"])
#individuals["Phone Number"].value_counts().index.to_list()

In [98]:
# Clean the phone numbers by removing quotations for now
def clean_phone(v):
    if pd.isna(v):
        return np.nan
    v = str(v)

    # Remove quotes and ., and replace , with ; separator
    v = re.sub(r"[\'’‘`´.]", "", v)
    v = re.sub(r"[,/]", ";", v)
    if v == "":
        return np.nan
    return v


In [99]:
individuals["Phone Number"] = individuals["Phone Number"].apply(clean_phone)

In [100]:
#individuals["Phone Number"].value_counts().index.to_list()

In [101]:
individuals["Phone Number"] = individuals["Phone Number"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Website</span>:
- The separator used was "and" which is less favourable to ; and there were also examples with whitespace unicodes that were actually duplicates, eg.:  'https://eng.mil.ru/en/index.htm\u202f\u202f\xa0' and 'https://eng.mil.ru/en/index.htm\u202f\u202f\u202f\xa0'

In [102]:
#profile_column(individuals, "Website", top_n=10, key=["SEID"])
#individuals["Website"].value_counts().index.to_list()

In [103]:
# Simple function applied to clean/format websites
def clean_website(v):
    if pd.isna(v):
        return np.nan

    # Lowercase standard
    v = str(v).strip().lower()

    # Replace word separator "and" with semicolon, and | too
    v = re.sub(r"\s+and\s+", "; ", v, flags=re.IGNORECASE)
    v = re.sub(r"\|", "; ", v)

    # Replace common symbol separators with ; (avoiding /)
    v = re.sub(r"\s*[&|,\n]\s*", "; ", v)

    # Remove spaces around . and /
    v = re.sub(r"\s*\.\s*", ".", v)
    v = re.sub(r"\s*/\s*", "/", v)
    if v == "":
        return np.nan

    return v

In [104]:
individuals["Website"] = individuals["Website"].apply(clean_website)
individuals["Website"] = individuals["Website"].apply(whitespace_fmt)

In [105]:
individuals["Website"].value_counts().index.to_list()

['http://www.amangroupco.com',
 'https://eng.mil.ru/en/index.htm',
 'www.sidar-dz.com',
 'pavel676@mail.ru',
 'https://savingpunjab.org/; https://www.instagram.com/lohadesigns/']

##### - <span style="color: #5d8eb8;">Email</span>:
- Here we have some mixed case emails, and some records with multiple emails are separated by **and** rather than ; like we'd prefer
- Note that the Email, Website and Phone number fields are sparse but may provide useful matching criteria so it's worth keeping

In [106]:
#profile_column(individuals, "Website", top_n=10, key=["SEID"])
#individuals["Email"].value_counts().index.to_list()

In [107]:
# Fix the email formatting issues
def clean_email(v):
    if pd.isna(v):
        return np.nan

    # Lowercase standard
    v = str(v).strip().lower()

    # Avoid websites, check emails are present via an @
    if "@" not in v:
        return np.nan

    # Remove spaces around @ and .
    v = re.sub(r"\s*@\s*", "@", v)
    v = re.sub(r"\s*\.\s*", ".", v)

    # Replace common separators and "and" with ;
    v = re.sub(r"\s+and\s+", "; ", v, flags=re.IGNORECASE)
    v = re.sub(r"\s*[&|,/\n]\s*", "; ", v)

    if v == "":
        return np.nan

    return v

In [108]:
individuals["Email"] = individuals["Email"].apply(clean_email)
individuals["Email"] = individuals["Email"].apply(whitespace_fmt)

In [109]:
#individuals["Email"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">National Identifier Number & Info</span>:
- This is understandably very poorly formatted, since different countries have different encodings for NIN, and may even allow the use of other IDs (eg. Taxpayer No.). We can still remove the apostrophe from some entries that (according to research) is unlikely to be deliberate
- There are NINs with special characters, like: Ф-703443 which make it hard to come up with a reliable regex
- Note that the NIN Info field can provide more context about some national numbers, and may also be used to impute missing nationalities (again, with an NLP/LLM approach for example)

In [110]:
#profile_column(individuals, "National Identifier Number", top_n=10, key=["SEID"])
#individuals["National Identifier Number"].value_counts().index.to_list()

In [111]:
# Format/Clean NIN (conservatively)
def clean_NIN(v):
    if pd.isna(v):
        return np.nan

    v = str(v).strip()
    v = re.sub(r"[\'’‘`´]", "", v)

    # Standardise upper casing??
    #v = v.upper()

    if v == "":
        return np.nan
    return v

In [112]:
individuals["National Identifier Number"] = individuals["National Identifier Number"].apply(clean_NIN)
individuals["National Identifier Number"] = individuals["National Identifier Number"].apply(whitespace_fmt)
individuals["National Identifier Info"] = individuals["National Identifier Info"].apply(whitespace_fmt)

In [113]:
#individuals["National Identifier Number"].value_counts().index.to_list()

In [114]:
#profile_column(individuals, "National Identifier Info", top_n=10, key=["SEID"])
#individuals["National Identifier Number"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">Passport Number & Info</span>:
- We face similar inconsistency issues here as with the NIN fields, since differnet countires have different systems, but some basic whitespace unicode formatting and removing unlikely characters like apostrophes again (eg. '06FBO2262 or '00085243) is good start (these kinds of characters are excluded for MRZ-Machine Readable Zone)

In [115]:
#profile_column(individuals, "Passport Number", top_n=10, key=["SEID"])
#individuals["Passport Number"].value_counts().index.to_list()

In [116]:
individuals["Passport Number"] = individuals["Passport Number"].apply(clean_NIN)
individuals["Passport Number"] = individuals["Passport Number"].apply(whitespace_fmt)
individuals["Passport Info"] = individuals["Passport Info"].apply(whitespace_fmt)

In [117]:
#individuals["Passport Number"].value_counts().index.to_list()

In [118]:
#profile_column(individuals, "Passport Info", top_n=10, key=["SEID"])
#individuals["Passport Info"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">Other Information, UK Statement of Reasons</span>:
- These are dense text fields that will only really benefit from whitespace reformatting
- Note that future work could use LLMs to impute missing information for each record from this text (where possible)

In [119]:
#profile_column(individuals, "UK Statement of Reasons", top_n=10, key=["SEID"])
#individuals[""].value_counts().index.to_list()

In [120]:
individuals["Other Information"] = individuals["Other Information"].apply(whitespace_fmt)
individuals["UK Statement of Reasons"] = individuals["UK Statement of Reasons"].apply(whitespace_fmt)

#### Save & Clear Memory

In [121]:
# Save and delete for memory
individuals.to_csv("individuals.csv", index=False)
del individuals
gc.collect()

0

### 3. entities.csv

In [122]:
entities = pd.read_csv("entities.csv")
entities.info()

<class 'pandas.DataFrame'>
RangeIndex: 29762 entries, 0 to 29761
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   SEID                       29762 non-null  str    
 1   LUID                       29762 non-null  str    
 2   Full Name                  29752 non-null  str    
 3   Name (Non-Latin Script)    2521 non-null   str    
 4   Non-Latin Script Type      1861 non-null   str    
 5   Non-Latin Script Language  1931 non-null   str    
 6   Name Type                  29756 non-null  str    
 7   Alias Strength             0 non-null      float64
 8   Address Lines              29762 non-null  str    
 9   Address Postal Code        9530 non-null   str    
 10  Address Country            28190 non-null  str    
 11  Phone Number               24568 non-null  str    
 12  Website                    19160 non-null  str    
 13  Email                      23767 non-null  str    
 14  B

#### Process each field

##### - <span style="color: #5d8eb8;">SEID, LUID</span>:
- No Processing Required (Clean)

In [123]:
#profile_column(entities, "LUID")
#entities["LUID"].value_counts().index.to_list()

In [124]:
entities["LUID"] = entities["LUID"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Full Name, Name (Non-Latin Script)</span>:
- When it comes to entities, short names are valid, typically as acronyms
- There are only 10 missing latin names, but they have corresponding Non-latin script names which can be used for matching instead

In [125]:
#profile_column(entities, "Full Name", key=["SEID"])
#entities[entities["SEID"] == "#0019689"]

In [126]:
# entities[
#     entities["Full Name"].isna()
# ]

In [127]:
entities["Full Name"] = entities["Full Name"].apply(whitespace_fmt)
entities["Name (Non-Latin Script)"] = entities["Name (Non-Latin Script)"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Non-Latin Script Type, Non-Latin Script Language</span>:
- Similar findings to that of individuals.csv
- We take a similar approach to data cleaning here 
- Note (The Non-Latin Script Language field is also appropriately formatted like previous datasets, but it contains many missing values)

In [128]:
#profile_column(entities, "Non-Latin Script Type", key=["SEID"])
#entities[entities["SEID"] == "#0019689"]

In [129]:
impute_seid = entities[ 
    (entities["Non-Latin Script Type"].isna()) & 
    (entities["Name (Non-Latin Script)"].notna()) 
]["SEID"]
mask = entities["SEID"].isin(impute_seid)
#impute_seid

In [130]:
entities[entities["SEID"].isin(impute_seid)][["Name (Non-Latin Script)", "Non-Latin Script Type"]].iloc[125:140,:]

,Name (Non-Latin Script),Non-Latin Script Type
1108,الملثمون,NaN
1109,الملثمون,NaN
1110,Les Enturbannés,NaN
1111,Les Enturbannés,NaN
1112,Les Enturbannés,NaN
1116,المرابطون,NaN
2112,الاتحاد الاسلامي,NaN
2113,شركة الكوثر للتوسط ببيع وشراء العملات الأجنبية,NaN
2117,القاعدة,NaN
2128,القاعده في العراق,NaN


In [131]:
entities.loc[mask, "Non-Latin Script Type"] = (
    entities.loc[mask, "Name (Non-Latin Script)"]
    .apply(detect_script)
)
entities[entities["SEID"].isin(impute_seid)][["Name (Non-Latin Script)", "Non-Latin Script Type"]].iloc[125:140,:]

,Name (Non-Latin Script),Non-Latin Script Type
1108,الملثمون,Arabic
1109,الملثمون,Arabic
1110,Les Enturbannés,NaN
1111,Les Enturbannés,NaN
1112,Les Enturbannés,NaN
1116,المرابطون,Arabic
2112,الاتحاد الاسلامي,Arabic
2113,شركة الكوثر للتوسط ببيع وشراء العملات الأجنبية,Arabic
2117,القاعدة,Arabic
2128,القاعده في العراق,Arabic


In [132]:
# entities[entities["SEID"].isin(impute_seid)&
#             (entities["Non-Latin Script Type"].isna())]

In [133]:
#profile_column(entities, "Non-Latin Script Language", top_n=18)

In [134]:
del impute_seid
del mask
gc.collect()

0

##### - <span style="color: #5d8eb8;">Name Type, Alias Strength</span>:
- Similar finding sin inconsistencies across name types, no ALias cases though, we can rename map as before
- Alias strength was actually not used, a full NaN column, and though it may be used in the future, we decide to drop it for now

In [135]:
entities["Name Type"] = entities["Name Type"].replace(name_type_map)

In [136]:
#profile_column(entities, "Alias Strength", top_n=10)

In [137]:
entities.drop(columns="Alias Strength", inplace=True)

In [138]:
#entities.info()

##### - <span style="color: #5d8eb8;">Address: Lines, Postal Code, Country</span>:
- We are going to have to reformat the Address Lines again here in a similar way to before
- After checking, no further processing is required

In [139]:
entities["Address Lines"] = entities["Address Lines"].apply(clean_address_lines)
entities["Address Postal Code"] = entities["Address Postal Code"].apply(clean_postcode)


In [140]:
#profile_column(entities, "Address Lines", top_n=10, key=["SEID"])
#entities["Address Country"].value_counts().index.to_list()

In [141]:
entities["Address Lines"] = entities["Address Lines"].apply(whitespace_fmt)
entities["Address Postal Code"] = entities["Address Postal Code"].apply(whitespace_fmt)
entities["Address Country"] = entities["Address Country"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Phone Number</span>:
- Similar formatting issues occured, but I noticed an entry with an "Unknown " value that can be set back to a NaN
- There are also some surprisingly short numbers like -774 with a - not a + as well, but I am not sure if that is useful information that shouldn't be ammended, or a typo (short emergency line numbers do exist)
- Also some numbers were separated by , or / which we can change to a ; (the poorly formatted values here guided us to improvements in our phone-cleaning function)

In [142]:
entities["Phone Number"] = entities["Phone Number"].apply(clean_phone)

In [143]:
#profile_column(entities, "Phone Number", top_n=10, key=["SEID"])
#entities["Phone Number"].value_counts().index.to_list()

In [144]:
entities["Phone Number"] = entities["Phone Number"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Website</span>:
- Found new examples where websites were separated by |, instead of our chosen ;

In [145]:
#profile_column(entities, "Website", top_n=10, key=["SEID"])
#entities["Website"].value_counts().index.to_list()

In [146]:
entities["Website"] = entities["Website"].apply(clean_website)
entities["Website"] = entities["Website"].apply(whitespace_fmt)

In [147]:
#entities["Website"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">Email</span>:
- Found a couple websites, eg. www.bloodandhonour.co.uk or http://www.petropars.com/fa/petroparsgroup/pre-شرکت-پتروپارس-ریسورسز-انجینیرینگ-PRE  instead of emails, which motivated a checking that there is an @ present
 

In [148]:
#profile_column(entities, "Email", top_n=10, key=["SEID"])
#entities["Email"].value_counts().index.to_list()

In [149]:
entities["Email"] = entities["Email"].apply(clean_email)
entities["Email"] = entities["Email"].apply(whitespace_fmt)

In [150]:
#entities["Email"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">Business Reg</span>:
- This field, similar to passport numbers and national identifier numbers, is poorly formatted likely due to different sources/encodings of business registrations. A conservative reformatting function as before is applied to avoid losing meaningful information

In [151]:
#profile_column(entities, "Business Reg", top_n=10, key=["SEID"])
#entities["Business Reg"].value_counts().index.to_list()

In [152]:
entities["Business Reg"] = entities["Business Reg"].apply(clean_NIN)
entities["Business Reg"] = entities["Business Reg"].apply(whitespace_fmt)

In [153]:
#entities["Business Reg"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">Type</span>:
- This field seems adequately formatted, just in need of some whitespace adjusting, eg. for 'Research Institute' and 'Research Institute '
- It doesn't seem to require multiple value per cells, and so no need to specify a consistent delimiter like ;
- Note: some fields may mean the same thing, eg. "Research Institute" and "Research", but this is ambiguous

In [154]:
#profile_column(entities, "Type", top_n=10, key=["SEID"])
#entities["Type"].value_counts().index.to_list()

In [155]:
entities["Type"] = entities["Type"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Subsidiaries</span>:
- Needs whitespace formatting
- Some examples like: '1. Sokan Arvand Ship Management Company 2. Persia Telecom Co. 3. Nilsun Kish 4. IOEC E&P ' are ambiguous in the sense that the ordering may contain meaningful information, so adjusting to a delimiter like ; over the numbered list could lose this information (it is not clear if all of the other unordered lists of parent companies delimited with a ; already follow the same ordering criteria)
- Some fields have additional spellings which can be delimted reliably, eg.: 'Sabai (Jewellery) Co., Ltd. (alternate spellings: Sabae (Gems and Jewellery) Co., Ltd., and Jasmine)'

In [156]:
#profile_column(entities, "Subsidiaries", top_n=10, key=["SEID"])
#entities["Subsidiaries"].value_counts().index.to_list()

In [157]:
def clean_subsid(v):
    if pd.isna(v):
        return np.nan
    v = str(v).strip()

    # Match: MAIN NAME (alternate spellings: ALTERNATES)
    match = re.match(
        r"^(.*?)\s*\(alternate spellings?:\s*(.*?)\)\s*$",
        v,
        flags=re.IGNORECASE
    )
    if match:
        main_name = match.group(1).strip()
        alternates = match.group(2).strip()
        v = f"{main_name}; {alternates}"

    if v == "":
        return np.nan

    return v

In [158]:
entities["Subsidiaries"] = entities["Subsidiaries"].apply(whitespace_fmt)
entities["Subsidiaries"] = entities["Subsidiaries"].apply(clean_subsid)

In [159]:
#entities["Subsidiaries"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">Parent Company</span>:
- This field only really needs whitespace format adjustment, though it does have an example like subsidiaries with a numbered list, eg: 'Ahdaf Investment Company has majority ownership (51.53%)1. AHDAF INVESTMENT COMPANY 2. GHADR INVESTMENT DEVELOPMENT CO. 3. SABA KARUN OIL AND GAS DEVELOPMENT COMPANY 4. SABA NAFT SUPPORT SERVICES COMPANY5. TOSSEE HAMI GHADR CO.'

In [160]:
#profile_column(entities, "Parent Company", top_n=10, key=["SEID"])
#entities["Parent Company"].value_counts().index.to_list()

In [161]:
entities["Parent Company"] = entities["Parent Company"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Other Information, UK Statement of Reasons</span>:

In [162]:
#profile_column(entities, "UK Statement of Reasons", top_n=10, key=["SEID"])
#individuals[""].value_counts().index.to_list()

In [163]:
entities["Other Information"] = entities["Other Information"].apply(whitespace_fmt)
entities["UK Statement of Reasons"] = entities["UK Statement of Reasons"].apply(whitespace_fmt)

#### Save & Clear Memory

In [164]:
# Save and delete for memory
entities.to_csv("entities.csv", index=False)
del entities
gc.collect()

0

### 4. ships.csv

In [165]:
ships = pd.read_csv("ships.csv")
ships.info()

<class 'pandas.DataFrame'>
RangeIndex: 857 entries, 0 to 856
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   SEID                       857 non-null    str    
 1   LUID                       857 non-null    str    
 2   Full Name                  857 non-null    str    
 3   Name (Non-Latin Script)    0 non-null      float64
 4   Non-Latin Script Type      0 non-null      float64
 5   Non-Latin Script Language  0 non-null      float64
 6   Name Type                  857 non-null    str    
 7   Alias Strength             0 non-null      float64
 8   IMO Number                 856 non-null    str    
 9   Current Believed Flag      725 non-null    str    
 10  Previous Flags             242 non-null    str    
 11  Type                       602 non-null    str    
 12  Tonnage                    402 non-null    float64
 13  Length                     246 non-null    float64
 14  Year 

#### Process each field

##### - <span style="color: #5d8eb8;">SEID, LUID</span>:
- No Processing Required (Clean)

In [166]:
ships["LUID"] = ships["LUID"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Full Name</span>:
- For ships, there were no missing names, with good textual formatting, there was an example: SEAGRACE / PACIFIC ALLIANCE where maybe applying a delimiter here could be ok, assuming the slash is not neccessary to the name, but rather it is dividing two separate names

In [167]:
ships.loc[ships["SEID"] == "#0048015", "Full Name"] = "SEAGRACE; PACIFIC ALLIANCE"

In [168]:
#profile_column(ships, "Full Name", key=["SEID"])
#ships["Full Name"].value_counts().index.to_list()

In [169]:
ships["Full Name"] = ships["Full Name"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Name (Non-Latin Script), Non-Latin Script Type, Non-Latin Script Language</span>:
- Whilst it is of course possible that these fields may be used in the future, the current UK sanctions data renders these null columns not worth keeping for now (for storage efficiency). We can always reintroduce them if needed later

In [170]:
ships.drop(columns=["Name (Non-Latin Script)", "Non-Latin Script Type", "Non-Latin Script Language"], inplace=True)

In [171]:
# ships["Full Name"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">Name Type, Alias Strength</span>:
- Similar findings in inconsistencies across name types, we can rename map as before. (althoight it's just to turn 48 instances of Primary name into proper Title Case)
- Alias strength was actually not used, a full NaN column, and though it may be used in the future, we decide to drop it for now

In [172]:
#profile_column(ships, "Name Type", top_n=10)

In [173]:
ships["Name Type"] = ships["Name Type"].replace(name_type_map)

In [174]:
#profile_column(ships, "Name Type", top_n=10)

In [175]:
ships.drop(columns="Alias Strength", inplace=True)

In [176]:
#ships.info()

##### - <span style="color: #5d8eb8;">IMO Number</span>:
- We found that the IMO numbers were either given as the 7 digit number or with an IMO in front of it to give 10 in character length. For formatting purposes, we can actually remove these IMOs and cast to an integer data type consistently

In [177]:
# Function to neatly format the IMO numbers
def clean_imo(v):
    if pd.isna(v):
        return np.nan

    v = str(v).strip().upper()

    # Remove IMO prefix if present
    v = re.sub(r"^IMO*", "", v)

    return v if v != "" else np.nan

In [178]:
ships["IMO Number"] = ships["IMO Number"].apply(clean_imo).astype("UInt32")

In [179]:
#profile_column(ships, "IMO Number", key=["SEID"])
#ships["IMO Number"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">Current Believed Flag, Previous Flags</span>:
- This is already well formatted data (Clean)
- There are quite a few missing values, but it is still informative and discerning information for our matches
- Note: No ships seem to have had multiple previous flags

In [180]:
#profile_column(ships, "Current Believed Flag", key=["SEID"])
#ships["Current Believed Flag"].value_counts().index.to_list()

In [181]:
#profile_column(ships, "Previous Flags", key=["SEID"])
#ships["Previous Flags"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">Type</span>:
- There are inconsistencies in the labelling of Oil Tanker and Oil tanker which can be fixed by renaming

In [182]:
#profile_column(ships, "Type", key=["SEID"], top_n=12)
#ships["Type"].value_counts().index.to_list()

In [183]:
ships["Type"] = ships["Type"].replace({"Oil tanker": "Oil Tanker"})
ships["Type"] = ships["Type"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Tonnage, Length, Year Built</span>:
- Formatted appropriately, but I may cast Tonnage to UInt32 since the decimal is unused, and also Year Built to UInt16, both helping save storage

In [184]:
#profile_column(ships, "Tonnage", key=["SEID"], top_n=12)

In [185]:
ships["Tonnage"] = ships["Tonnage"].astype("UInt32")
ships["Year Built"] = ships["Year Built"].astype("UInt16")

In [186]:
#profile_column(ships, "Year Built", key=["SEID"], top_n=12)
#ships["Type"].value_counts().index.to_list()

##### - <span style="color: #5d8eb8;">Other Information, UK Statement of Reasons</span>:

In [187]:
#profile_column(ships, "UK Statement of Reasons", top_n=10, key=["SEID"])
#ships["UK Statement of Reasons"].value_counts().index.to_list()

In [188]:
ships["Other Information"] = ships["Other Information"].apply(whitespace_fmt)
ships["UK Statement of Reasons"] = ships["UK Statement of Reasons"].apply(whitespace_fmt)

#### Save & Clear Memory

In [189]:
# Save and delete for memory
ships.to_csv("ships.csv", index=False)
del ships
gc.collect()

0

### 5. sanctions.csv

In [190]:
sanctions = pd.read_csv("sanctions.csv")
sanctions.info()

<class 'pandas.DataFrame'>
RangeIndex: 57033 entries, 0 to 57032
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   SEID               57033 non-null  str  
 1   Regime Name        57033 non-null  str  
 2   Sanctions Imposed  57033 non-null  str  
 3   Date Designated    57033 non-null  str  
 4   Last Updated       57033 non-null  str  
dtypes: str(5)
memory usage: 2.2 MB


#### Process each field

##### - <span style="color: #5d8eb8;">SEID</span>:
- No Processing Required (Clean)

##### - <span style="color: #5d8eb8;">Regime Name</span>:
- The 31 different sanction regime names are well formatted already

In [191]:
#profile_column(sanctions, "Regime Name", key=["SEID"])
#sanctions["Regime Name"].value_counts().index.to_list()

In [192]:
sanctions["Regime Name"] = sanctions["Regime Name"].apply(whitespace_fmt)

##### - <span style="color: #5d8eb8;">Sanctions Imposed</span>:
- These cells have multiple values, which seem to be listing the different types of sanctions imposed. We can explode these out to preserve better atomicity
- There are a couple of mis-formatted value types, eg: "Prohibition on Correspondent banking relationships and processing payments" and "Prohibition on correspondent banking relationships and processing payments"

In [193]:
#profile_column(sanctions, "Sanctions Imposed", key=["SEID"])

In [194]:
sanctions["Sanctions Imposed"] = (
    sanctions["Sanctions Imposed"]
    .str.split(r"\s*[|]\s*", regex=True)     # Only need to split on | here
)   

# Explode
sanctions = sanctions.explode("Sanctions Imposed", ignore_index=True)

In [195]:
sanctions["Sanctions Imposed"] = sanctions["Sanctions Imposed"].replace({
    "Prohibition on Correspondent banking relationships and processing payments": "Prohibition on correspondent banking relationships and processing payments"
})

In [196]:
# Turn the odd sanction into a consistent format
def clean_sanctions(v):
    if pd.isna(v):
        return np.nan
    v = str(v)
    v = re.sub(
        r"prohibition on correspondent banking relationships and processing payments",
        "Prohibition on correspondent banking relationships and processing payments",
        v,
        flags=re.IGNORECASE
    )

    return v if v else np.nan

In [197]:
sanctions["Sanctions Imposed"] = sanctions["Sanctions Imposed"].apply(whitespace_fmt)
sanctions["Sanctions Imposed"] = sanctions["Sanctions Imposed"].apply(clean_sanctions)

In [198]:
#profile_column(sanctions, "Sanctions Imposed", key=["SEID"])
#sanctions["Sanctions Imposed"].value_counts().index.to_list()
sanctions[sanctions["SEID"] == "#0044008"]

,SEID,Regime Name,Sanctions Imposed,Date Designated,Last Updated
102324,#0044008,The Russia (Sanctions) (EU Exit) Regulations 2019,Asset freeze,22/02/2022,09/04/2025
102325,#0044008,The Russia (Sanctions) (EU Exit) Regulations 2019,Trust Services Sanctions,22/02/2022,09/04/2025
102326,#0044008,The Russia (Sanctions) (EU Exit) Regulations 2019,Director Disqualification Sanction,22/02/2022,09/04/2025
102327,#0044008,The Russia (Sanctions) (EU Exit) Regulations 2019,Prohibition on correspondent banking relations...,22/02/2022,09/04/2025
102328,#0044008,The Russia (Sanctions) (EU Exit) Regulations 2019,Prohibition on correspondent banking relations...,22/02/2022,09/04/2025


##### - <span style="color: #5d8eb8;">Date Designated, Last Updated</span>:
- These dates were already formatted correctly, so we cast them to datetime datatypes

In [199]:
#profile_column(sanctions, "Last Updated", key=["SEID"])
#sanctions["Sanctions Imposed"].value_counts().index.to_list()

In [200]:
# Check date formats across strings
for col in ["Date Designated", "Last Updated"]:
    temp = pd.to_datetime(sanctions["Last Updated"], errors="coerce", dayfirst=False)
    print(len(sanctions[sanctions["Last Updated"].notna() & temp.isna()]))
del temp
gc.collect()

0
0


C:\Users\alecz\AppData\Local\Temp\ipykernel_23480\3790393128.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  temp = pd.to_datetime(sanctions["Last Updated"], errors="coerce", dayfirst=False)
C:\Users\alecz\AppData\Local\Temp\ipykernel_23480\3790393128.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  temp = pd.to_datetime(sanctions["Last Updated"], errors="coerce", dayfirst=False)


0

In [201]:
for col in ["Date Designated", "Last Updated"]:
    sanctions[col] = pd.to_datetime(sanctions[col], dayfirst=False)

C:\Users\alecz\AppData\Local\Temp\ipykernel_23480\637305047.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sanctions[col] = pd.to_datetime(sanctions[col], dayfirst=False)
C:\Users\alecz\AppData\Local\Temp\ipykernel_23480\637305047.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sanctions[col] = pd.to_datetime(sanctions[col], dayfirst=False)


In [202]:
#profile_column(sanctions, "Date Designated", key=["SEID"])

#### Save & Clear Memory

In [203]:
# Save and delete for memory
sanctions.to_csv("sanctions.csv", index=False)
del sanctions
gc.collect()

0

### 6. Cross-dataset processing & Checks

This is where we load and remove the rows across & between each dataset. In our case, the only rows we decided should be removed were those instances of a Haji which was mostly NaN across all fields

In [204]:
for ds in ["name_index", "individuals", "entities", "ships", "sanctions"]:
    df = pd.read_csv(f"{ds}.csv")
    df = df[~(
        df["SEID"].isin(rm_seids)
    )]
    df.to_csv(f"{ds}.csv", index=False)

del df
gc.collect()

0

In [205]:
# View Some Records
# df = pd.read_csv("individuals.csv")
# df.iloc[random.randint(0, len(df)-1),:]

## Duplicates

Typically, check the clean transformed data for any duplicates. We start by identifying duplicate records in individuals, entities and ships .csv first. We can then use those findings to deduplicate name_index and sanctions, and then carry out any further record deduplication as necessary (eg in sanctions, the same sanction imposed to the same SEID on the same date of designation)

In [235]:
"""
Finds possible duplicate rows using fuzzy similarity across multiple columns.
A pair of rows is flagged only if every selected column has a similarity score
greater than or equal to the corresponding threshold.
"""
def identify_duplicates(
    df: pd.DataFrame,          # DataFrame to search through
    cols: list[str],           # List of fields determining duplicate criteria
    threshes: list[int],       # The similarity thresholds for each field
    k_cols: str | list[str],   # The primary key fields
    match_NaN: bool = False    # Allows Matches on NaNs 
) -> pd.DataFrame:
    
    # Standardise Input (Compound?) Primary key
    if isinstance(k_cols, str):
        k_cols = [k_cols]

    # Sanitise Input Lengths
    if len(cols) != len(threshes):
        raise ValueError("Columns and Thresholds must have the same length!")

    results = []
    compare_df = df[k_cols + cols].copy()

    # Standardise/Prepare cols for fuzzy matching
    for col in cols:
        compare_df[col] = (
            compare_df[col]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    # Make a list of dictionaries from the rows
    rows = compare_df.to_dict("records")

    for i in tqdm(range(len(rows)), desc="Checking Pairs"):
        for j in range(i + 1, len(rows)):
            scores = {}
            is_match = True

            for col, thresh in zip(cols, threshes):
                val1 = rows[i][col]
                val2 = rows[j][col]

                # If both values are missing/blank, decide what to do on matchNaN parameter
                if val1 == "" and val2 == "":
                    if match_NaN:
                        score = 100.0
                    else:
                        score = np.nan
                        is_match = False
                else:
                    # Otherwise, compute similarity score
                    score = fuzz.token_sort_ratio(val1, val2)
                    if score < thresh:
                        is_match = False

                scores[f"{col}_score"] = score

            if is_match:
                result = {}

                # Add key fields for both rows
                for col in k_cols:
                    result[f"{col}_1"] = rows[i][col]
                    result[f"{col}_2"] = rows[j][col]

                # Add compared values for both rows
                for col in cols:
                    result[f"{col}_1"] = rows[i][col]
                    result[f"{col}_2"] = rows[j][col]

                # Add similarity scores and add the whole result to result tracker
                result.update(scores)
                results.append(result)

    return pd.DataFrame(results)

In [243]:
#individuals = pd.read_csv("individuals.csv")
individuals = load_csv_chunk("individuals.csv", 10, 0)

In [244]:
individuals.info()

<class 'pandas.DataFrame'>
RangeIndex: 2638 entries, 0 to 2637
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   SEID                        2638 non-null   str    
 1   LUID                        2638 non-null   str    
 2   Surname                     2636 non-null   str    
 3   Given Names                 2358 non-null   str    
 4   Name (Non-Latin Script)     864 non-null    str    
 5   Non-Latin Script Type       864 non-null    str    
 6   Non-Latin Script Language   14 non-null     str    
 7   Name Type                   2638 non-null   str    
 8   Alias Strength              1534 non-null   str    
 9   D.O.B                       2638 non-null   str    
 10  Gender                      1186 non-null   str    
 11  Title                       1131 non-null   str    
 12  Position                    1534 non-null   str    
 13  Nationalities               2443 non-null   

In [ ]:
# If you overspecify fields, may miss a matching with this algorithm if one record has it filled 
# and the other doesnt. But on the other hand, if you underspecify, you may match two different people as being the same person
# 

# Strike a balance

In [255]:
res = identify_duplicates(
    individuals, 
    ["Surname", "Given Names", "Name (Non-Latin Script)", "D.O.B", 
     "Gender", "Nationalities", "National Identifier Number", "Passport Number"],
    [95, 95, 95, 100, 100, 95, 95, 95],
    "SEID",
    match_NaN=True
)

Checking Pairs: 100%|██████████| 2638/2638 [00:17<00:00, 151.80it/s]


In [256]:
res

,SEID_1,SEID_2,Surname_1,Surname_2,Given Names_1,Given Names_2,Name (Non-Latin Script)_1,Name (Non-Latin Script)_2,D.O.B_1,D.O.B_2,...,Passport Number_1,Passport Number_2,Surname_score,Given Names_score,Name (Non-Latin Script)_score,D.O.B_score,Gender_score,Nationalities_score,National Identifier Number_score,Passport Number_score
0,#0001061,#0001062,akhund,akhund,mohammad hassan,mohammad hassan,محمد حسن آخوند,محمد حسن آخوند,dd/mm/1950,dd/mm/1950,...,p04581926,p04581926,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,#0001061,#0001097,akhund,akhund,mohammad hassan,mohammad hassan,محمد حسن آخوند,محمد حسن آخوند,dd/mm/1950,dd/mm/1950,...,p04581926,p04581926,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2,#0001061,#0001098,akhund,akhund,mohammad hassan,mohammad hassan,محمد حسن آخوند,محمد حسن آخوند,dd/mm/1950,dd/mm/1950,...,p04581926,p04581926,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
3,#0001062,#0001097,akhund,akhund,mohammad hassan,mohammad hassan,محمد حسن آخوند,محمد حسن آخوند,dd/mm/1950,dd/mm/1950,...,p04581926,p04581926,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
4,#0001062,#0001098,akhund,akhund,mohammad hassan,mohammad hassan,محمد حسن آخوند,محمد حسن آخوند,dd/mm/1950,dd/mm/1950,...,p04581926,p04581926,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4904,#0005194,#0005195,ahmad,ahmad,hasan,hasan,,,03/11/1957,03/11/1957,...,,,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
4905,#0005236,#0005237,al-masli,al-masli,abd-al-hamid,abd-al-hamid,عبدالحميد المصلي,عبدالحميد المصلي,dd/mm/1976,dd/mm/1976,...,,,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
4906,#0005238,#0005239,al-darnavi,al-darnavi,hamza,hamza,,,dd/mm/1976,dd/mm/1976,...,,,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
4907,#0005240,#0005241,al-darnawi,al-darnawi,abu-hamzah,abu-hamzah,,,dd/mm/1976,dd/mm/1976,...,,,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0


In [249]:


def group_duplicates(
    dupe_pairs: pd.DataFrame,
    id_col_1: str = "SEID_1",
    id_col_2: str = "SEID_2"
) -> list[list[Any]]:
    """
    Convert pairwise duplicate matches into connected duplicate groups.

    This is designed to work with the output of find_possible_duplicates(),
    where duplicate pairs are represented by two ID columns, e.g. SEID_1 and SEID_2.

    Parameters
    ----------
    dupe_pairs : pd.DataFrame
        DataFrame containing pairwise duplicate matches.

    id_col_1 : str
        Column containing the first ID in each duplicate pair.

    id_col_2 : str
        Column containing the second ID in each duplicate pair.

    Returns
    -------
    list[list[Any]]
        A list of duplicate groups. Each group is a list of connected IDs.
    """

    required_cols = [id_col_1, id_col_2]
    missing_cols = [col for col in required_cols if col not in dupe_pairs.columns]

    if missing_cols:
        raise ValueError(f"Missing columns from dupe_pairs: {missing_cols}")

    if dupe_pairs.empty:
        return []

    graph: dict[Any, set[Any]] = defaultdict(set)

    for _, row in dupe_pairs[[id_col_1, id_col_2]].dropna().iterrows():
        id_1 = row[id_col_1]
        id_2 = row[id_col_2]

        if id_1 == id_2:
            continue

        graph[id_1].add(id_2)
        graph[id_2].add(id_1)

    visited: set[Any] = set()
    groups: list[list[Any]] = []

    for node in graph:
        if node in visited:
            continue

        queue: deque[Any] = deque([node])
        visited.add(node)

        group: list[Any] = []

        while queue:
            current = queue.popleft()
            group.append(current)

            for neighbour in graph[current]:
                if neighbour not in visited:
                    visited.add(neighbour)
                    queue.append(neighbour)

        groups.append(sorted(group))

    return groups

In [250]:
duplicate_groups = group_duplicates(
    res,
    id_col_1="SEID_1",
    id_col_2="SEID_2"
)

In [257]:
len(duplicate_groups)

570

In [263]:
individuals[
    individuals["SEID"].isin(duplicate_groups[334])
]

,SEID,LUID,Surname,Given Names,Name (Non-Latin Script),Non-Latin Script Type,Non-Latin Script Language,Name Type,Alias Strength,D.O.B,...,Address Country,Phone Number,Website,Email,National Identifier Number,National Identifier Info,Passport Number,Passport Info,Other Information,UK Statement of Reasons
1219,#0002280,AFG0092,Hamidullah,Saeed,NaN,NaN,NaN,Alias,Good quality a.k.a,dd/mm/1973,...,Afghanistan,NaN,NaN,NaN,NaN,NaN,D000972185,Afghanistan diplomatic passport; issued 12 Oct...,Belongs to Ghilzai tribe. Height: 175cm. Revie...,NaN
1220,#0002281,AFG0092,Hamidullah,Saeed,NaN,NaN,NaN,Alias,Good quality a.k.a,dd/mm/1973,...,Afghanistan,NaN,NaN,NaN,NaN,NaN,D000972185,Afghanistan diplomatic passport; issued 12 Oct...,Belongs to Ghilzai tribe. Height: 175cm. Revie...,NaN
1221,#0002282,AFG0092,Hamidullah,Saeed,NaN,NaN,NaN,Alias,Good quality a.k.a,dd/mm/1973,...,Afghanistan,NaN,NaN,NaN,NaN,NaN,D000972185,Afghanistan diplomatic passport; issued 12 Oct...,Belongs to Ghilzai tribe. Height: 175cm. Revie...,NaN


In [251]:
duplicate_groups

[['#0001061', '#0001062', '#0001097', '#0001098'],
 ['#0001063', '#0001064', '#0001065', '#0001066'],
 ['#0001067', '#0001068', '#0001069', '#0001070'],
 ['#0001071', '#0001072', '#0001073', '#0001074'],
 ['#0001075', '#0001076', '#0001077', '#0001078'],
 ['#0001079', '#0001080', '#0001081', '#0001082'],
 ['#0001083', '#0001084', '#0001099', '#0001100'],
 ['#0001085', '#0001086', '#0001087', '#0001088'],
 ['#0001089', '#0001090', '#0001091', '#0001092'],
 ['#0001093', '#0001094', '#0001095', '#0001096'],
 ['#0001101', '#0001102', '#0001103'],
 ['#0001104', '#0001105', '#0001106'],
 ['#0001107', '#0001108', '#0001109'],
 ['#0001110', '#0001111', '#0001112'],
 ['#0001113', '#0001114', '#0001115'],
 ['#0001116', '#0001117', '#0001118'],
 ['#0001119', '#0001120', '#0001121'],
 ['#0001122', '#0001123', '#0001124'],
 ['#0001125', '#0001126', '#0001131', '#0001132'],
 ['#0001127', '#0001128', '#0001133', '#0001134'],
 ['#0001129', '#0001130', '#0001135', '#0001136'],
 ['#0001137', '#0001138']

In [ ]:
def create_seid_map(
    duplicate_groups: list[list[Any]]
) -> dict[Any, Any]:
    """
    Create a mapping from duplicate SEIDs to canonical SEIDs.
    """

    seid_map = {}

    for group in duplicate_groups:
        group = sorted(group)
        canonical_id = group[0]

        for duplicate_id in group[1:]:
            seid_map[duplicate_id] = canonical_id

    return seid_map

In [ ]:
def merge_duplicate_groups(
    df: pd.DataFrame,
    duplicate_groups: list[list[Any]],
    id_col: str = "SEID",
    separator: str = " ; "
) -> pd.DataFrame:
    """
    Merge duplicate records into one consolidated record per duplicate group.

    For each group:
    - the lowest ID is kept as the canonical ID
    - non-missing values are consolidated across rows
    - differing values are joined with a separator
    - duplicate rows are removed

    Parameters
    ----------
    df : pd.DataFrame
        Dataset containing records to deduplicate.

    duplicate_groups : list[list[Any]]
        Duplicate groups, e.g. [["#000001", "#000004"], ["#000020", "#000021"]].

    id_col : str
        ID column used to identify records.

    separator : str
        Separator used when multiple distinct values exist for a field.

    Returns
    -------
    pd.DataFrame
        Deduplicated DataFrame with consolidated records.
    """

    if id_col not in df.columns:
        raise ValueError(f"{id_col} not found in DataFrame.")

    df_out = df.copy()

    rows_to_drop = []

    for group in duplicate_groups:
        group = sorted(group)

        canonical_id = group[0]
        duplicate_ids = group[1:]

        group_rows = df_out[df_out[id_col].isin(group)]

        if group_rows.empty:
            continue

        consolidated = {}

        for col in df_out.columns:
            if col == id_col:
                consolidated[col] = canonical_id
                continue

            values = (
                group_rows[col]
                .dropna()
                .astype(str)
                .str.strip()
            )

            values = values[values != ""]
            unique_values = list(dict.fromkeys(values))

            if len(unique_values) == 0:
                consolidated[col] = np.nan
            elif len(unique_values) == 1:
                consolidated[col] = unique_values[0]
            else:
                consolidated[col] = separator.join(unique_values)

        # Replace canonical row with consolidated row
        canonical_index = df_out[df_out[id_col] == canonical_id].index[0]

        for col, value in consolidated.items():
            df_out.at[canonical_index, col] = value

        # Mark duplicate rows for removal
        rows_to_drop.extend(
            df_out[df_out[id_col].isin(duplicate_ids)].index.tolist()
        )

    df_out = df_out.drop(index=rows_to_drop)

    return df_out.reset_index(drop=True)

## QC

Quality control, check if theres any missing values, any obvious duplicates, the range/values fields take, etc...

This is where you do a discussion thing again essentially

## Export

In [ ]:
"""Put it all in a clean data folder, os remove the other stuff in current dir"""